# Flipkart Gridlock Hackathon 2.0 — Solution Approach

This notebook collates the full solution writeup with live result cells that read actual pipeline artifacts.
All source files are in `../src/`. Run `python ../src/main.py` to regenerate artifacts.

**Sections**
1. [Overview](#1-overview)
2. [Data & Cross-Validation](#2-data--cross-validation)
3. [Feature Engineering](#3-feature-engineering)
4. [Models & Ensemble](#4-models--ensemble)
5. [Results & Lessons](#5-results--lessons)

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd

ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
ARTIFACTS = os.path.join(ROOT, 'artifacts')
LOGS      = os.path.join(ROOT, 'logs')
DATA      = os.path.join(ROOT, 'data')
SRC       = os.path.join(ROOT, 'src')
sys.path.insert(0, SRC)

def load_json(name):
    with open(os.path.join(ARTIFACTS, name)) as f:
        return json.load(f)

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try: rows.append(json.loads(line))
                except: pass
    return rows

print('Root:', ROOT)

---
## 1 Overview

> *Full writeup: [01_overview.md](01_overview.md)*

**Task** — predict normalised traffic demand (`demand` ∈ (0, 1]) for 41,778 day-49 test rows.  
**Metric** — `max(0, 100 × R²)` on original demand scale.  
**Core insight** — day-48 demand at the same geohash and time slot is the strongest single predictor of day-49 demand.

### Pipeline
```
data.py → preprocessing.py → features.py → model.py → reporting.py → evaluation.py
```

### Tools
| Tool | Role |
|------|------|
| LightGBM | GBDT with smoothed target-encoded geohash |
| CatBoost | GBDT with native raw-geohash encoding |
| ExtraTrees | Randomized ensemble — best single-model OOF |
| XGBoost | Additional GBDT member |
| scipy SLSQP | Exact blend weight optimization on simplex |
| geohash2 | Geohash → (lat, lon) decode |
| sklearn | NearestNeighbors (cold-start), KMeans (clusters) |
| pytest | 46-test suite: leakage guards + quality floors |

In [ ]:
train = pd.read_csv(os.path.join(DATA, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA, 'test.csv'))

print(f'Train: {train.shape}   Test: {test.shape}')
print(f'Unique geohashes (train): {train["geohash"].nunique()}')
print(f'Days in train: {sorted(train["day"].unique())}')
print()
print('Demand distribution:')
print(train['demand'].describe().round(4))

---
## 2 Data & Cross-Validation

> *Full writeup: [02_data_and_cv.md](02_data_and_cv.md)*

### CV Design
```
Day 48  (69,427 rows)  →  fold = -1   training
Day 49  ( 7,872 rows)  →  fold =  0   validation
```

### Target Transformation
Train on `log(demand)` to handle right-skew; invert with `exp()` + clip to [1e-6, 1.0].

### Nighttime / Midday Mismatch
The val fold covers **hours 0–2 only**; test is **hour 3 onward**. Any global calibration fit on the nighttime fold overcorrects midday test predictions — calibration is disabled.

In [ ]:
folds = np.load(os.path.join(ARTIFACTS, 'folds.npy'))
y_log = np.load(os.path.join(ARTIFACTS, 'y_log.npy'))

n_train = (folds == -1).sum()
n_val   = (folds ==  0).sum()
print(f'Fold -1 (train): {n_train:,} rows')
print(f'Fold  0 (val):   {n_val:,} rows')
print()

demand = np.exp(y_log)
print(f'Train fold demand — mean: {demand[folds==-1].mean():.4f}  median: {np.median(demand[folds==-1]):.4f}')
print(f'Val   fold demand — mean: {demand[folds== 0].mean():.4f}  median: {np.median(demand[folds== 0]):.4f}')
print()
print('(Val fold is nighttime only — lower demand than overall train)')

---
## 3 Feature Engineering

> *Full writeup: [03_feature_engineering.md](03_feature_engineering.md)*

**46 numeric + 4 categorical features** (`geohash`, `RoadType`, `Weather`, `geohash_cluster`).

| Group | Count | Key features |
|-------|-------|--------------|
| Time | 8 | `minute_of_day`, cyclic harmonics (k=1,2,3) |
| Spatial | 2 | `lat`, `lon` from geohash decode |
| Road & infra | 3 | `NumberofLanes`, `large_vehicles`, `landmarks` |
| Weather | 4 | `Temperature` (imputed), 3 missing-value flags |
| Day-48 carry-forward | 27 | same-slot, distribution shape, spatial neighbors, dynamics, multi-lag, rank |
| Day-49 autoregressive | 2 | `demand_d49_morning_mean` (LOO), `demand_d49_last_known` |
| Target encodings | 3 | geohash TE, prefix TE, (geohash, hour) TE |

**Self-reference fix:** day-48 training rows would perfectly predict themselves via same-slot lookup. Fix: those rows receive the mean of ±15-min neighbor slots as a proxy. Day-49 and test rows get the genuine cross-day lookup.

In [ ]:
metrics = load_json('metrics.json')

if 'feature_importance' in metrics:
    imp = pd.DataFrame(metrics['feature_importance'], columns=['feature', 'gain'])
    print('Top-20 LightGBM features by gain:')
    print(imp.head(20).to_string(index=False))
else:
    print('feature_importance not in metrics.json — run src/main.py to regenerate')
    print('Available keys:', list(metrics.keys()))

In [ ]:
# Show OOF prediction correlations between models (diversity check)
lgbm_oof = np.load(os.path.join(ARTIFACTS, 'oof_lgbm.npy'))
cat_oof  = np.load(os.path.join(ARTIFACTS, 'oof_cat.npy'))
et_oof   = np.load(os.path.join(ARTIFACTS, 'oof_et.npy'))

val_mask = folds == 0
corr_lc = np.corrcoef(lgbm_oof[val_mask], cat_oof[val_mask])[0, 1]
corr_le = np.corrcoef(lgbm_oof[val_mask], et_oof[val_mask])[0, 1]
corr_ce = np.corrcoef(cat_oof[val_mask],  et_oof[val_mask])[0, 1]

print('OOF prediction correlations (val fold):')
print(f'  LGBM vs CatBoost:    {corr_lc:.4f}')
print(f'  LGBM vs ExtraTrees:  {corr_le:.4f}')
print(f'  CatBoost vs ET:      {corr_ce:.4f}')
print()
print('Lower = more diverse = more blending benefit')

---
## 4 Models & Ensemble

> *Full writeup: [04_models_and_ensemble.md](04_models_and_ensemble.md)*

### Model Summary
| Model | Key config | Notes |
|-------|-----------|-------|
| LightGBM | `num_leaves=63`, `lambda_l2=3`, no raw geohash | Drops geohash to force diversity vs CatBoost |
| CatBoost | `depth=6`, `l2_leaf_reg=5`, native geohash | `depth=8` + large feature set → overfitting |
| ExtraTrees | `n_estimators=600`, `max_features=0.6`, `min_samples_leaf=20` | Best OOF; robust to feature dilution |
| XGBoost | `max_depth=7`, `subsample=0.8` | In ensemble; 0% blend weight currently |

### Blend Optimizer
Scipy SLSQP: maximise `R²(blend) − 0.01 × Σw²` subject to `w ≥ 0`, `Σw = 1`.  
Multiple restarts (uniform + one-hot inits) avoid local optima.

In [ ]:
from sklearn.metrics import r2_score

def r2_demand(y_log, pred_log):
    return r2_score(np.clip(np.exp(y_log), 1e-6, 1.0),
                    np.clip(np.exp(pred_log), 1e-6, 1.0))

y_val  = y_log[val_mask]

try:
    xgb_oof = np.load(os.path.join(ARTIFACTS, 'oof_xgb.npy'))
    models = {'LightGBM': lgbm_oof, 'CatBoost': cat_oof, 'ExtraTrees': et_oof, 'XGBoost': xgb_oof}
except FileNotFoundError:
    models = {'LightGBM': lgbm_oof, 'CatBoost': cat_oof, 'ExtraTrees': et_oof}

print(f'{"Model":<12}  {"OOF R²":>8}  {"Val demand mean":>16}')
print('-' * 42)
for name, oof in models.items():
    if not np.all(np.isnan(oof[val_mask])):
        r2 = r2_demand(y_val, oof[val_mask])
        pred_mean = np.clip(np.exp(oof[val_mask]), 1e-6, 1.0).mean()
        print(f'{name:<12}  {r2:>8.4f}  {pred_mean:>16.4f}')

print()
true_mean = np.clip(np.exp(y_val), 1e-6, 1.0).mean()
print(f'True val demand mean: {true_mean:.4f}')

In [ ]:
# Load blend weights from latest run
runs = load_jsonl(os.path.join(LOGS, 'runs.jsonl'))
latest = runs[-1]

w = latest.get('w', {})
print('Latest blend weights:')
for model, weight in w.items():
    bar = '█' * int(weight * 30)
    print(f'  {model:<12} {weight:.3f}  {bar}')
print()
print(f'Blended OOF R²: {latest.get("blend_calib", latest.get("blend_raw", "?")):.4f}')
print(f'Calibration:    a={latest.get("calib_ab", [1,0])[0]:.3f}  b={latest.get("calib_ab", [1,0])[1]:.3f}')

---
## 5 Results & Lessons

> *Full writeup: [05_results.md](05_results.md)*

In [ ]:
# Leaderboard history from runs.jsonl
rows = []
for r in runs:
    rows.append({
        'timestamp':  r.get('ts', '')[:16],
        'n_features': r.get('n_feat', '?'),
        'ET OOF R²':  r.get('et', r.get('blend_calib', '?')),
        'LGBM OOF R²': r.get('lgbm', '?'),
        'LB score':   r.get('lb', '—'),
    })

df_runs = pd.DataFrame(rows)
print(df_runs.to_string(index=False))

In [ ]:
# Post-blend diagnostics from eval.json
try:
    ev = load_json('eval.json')

    print('=== Per-bucket diagnostics (val fold) ===')
    bm = ev.get('bucket_metrics', {})
    print(f'{"Bucket":<22}  {"n":>6}  {"R²":>7}  {"MAE":>7}  {"Acc":>6}')
    print('-' * 55)
    for label, stats in bm.items():
        print(f'{label:<22}  {stats["n"]:>6}  {stats["r2"]:>7.4f}  {stats["mae"]:>7.4f}  {stats["acc"]:>6.3f}')

    print()
    print('=== Spatial summary ===')
    sp = ev.get('spatial', {})
    print(f'Log-MAE mean:  {sp.get("mae_mean", sp.get("r2_mean", "?")):.4f}')
    print(f'Worst geohash: {sp.get("worst_5", [{}])[0]}')
    print(f'Cold-start:    {ev.get("cold_start_count", "?")} test geohashes')

    print()
    print('=== Test prediction distribution ===')
    td = ev.get('test_pred_dist', {})
    print(f'mean={td.get("mean","?"):.4f}  std={td.get("std","?"):.4f}  '
          f'p50={td.get("p50","?"):.4f}  p90={td.get("p90","?"):.4f}')

except FileNotFoundError:
    print('artifacts/eval.json not found — run src/main.py first')

In [ ]:
# Preview final submission
sub_path = os.path.join(ROOT, 'submissions', 'submission.csv')
if os.path.exists(sub_path):
    sub = pd.read_csv(sub_path)
    print(f'Submission shape: {sub.shape}')
    print(f'Demand — min: {sub["demand"].min():.6f}  max: {sub["demand"].max():.6f}')
    print(f'         mean: {sub["demand"].mean():.4f}  std: {sub["demand"].std():.4f}')
    print()
    print(sub.head(10).to_string(index=False))
else:
    print('No submission.csv found — run src/main.py')

## Key Lessons

**1. Calibration must be time-stratified.**  
A global log-shift fit on the nighttime OOF fold (hours 0–2) overcorrects midday test predictions by ≈1.32×.  
Regression: LB 87.46 → 81.89 with CALIB_SHRINK=0.7. Fix: CALIB_SHRINK=0.0.

**2. Feature dilution hurts ExtraTrees.**  
Adding 14 new features (61 total) reduced ET OOF from 0.6355 → 0.6321 and LB from 87.46 → 86.92.  
Ablating to the top-3 new features recovered OOF to 0.6337.

**3. LGBM diversity depends on `num_leaves`.**  
`num_leaves=127` caused LGBM to early-stop at ~50 iters and CatBoost OOF to collapse.  
`num_leaves=63` restores stable GBDT convergence.

**4. OOF R² is a relative signal, not absolute ground truth.**  
The 7,872-row nighttime val fold is too narrow to evaluate models in midday regimes.  
All feature and calibration decisions require LB confirmation.